# The Z–hadron wake in Pb+Pb: Z+jet events

The jet-jet wake studies (`wake_hadrons.ipynb`, `jet_wake.ipynb`) measure the wake around the
**leading parton**. In a Z+jet event the reference is the **Z**: it leaves the medium untouched,
so its direction is the recoiling parton's before any quenching, and the wake can be looked for
in angle *and* rapidity relative to it, as CMS
([2507.09307](https://arxiv.org/abs/2507.09307), [2103.04377](https://arxiv.org/abs/2103.04377))
and ATLAS ([2008.09811](https://arxiv.org/abs/2008.09811)) do:

* **Δφ = φ_h − φ_Z**: the recoiling jet sits at Δφ ≈ π, the Z side at Δφ ≈ 0;
* **Δη = η_h − y_Z**: the hadron's pseudorapidity against the Z's rapidity.

What a hydrodynamic wake predicts: the jet pushes the medium ahead of it (a **positive wake**,
extra soft hadrons around Δφ ≈ π) and leaves a depleted region behind (the **diffusion wake**,
fewer soft hadrons around Δφ ≈ 0 and Δη ≈ 0). CMS's evidence for the latter is a dip at
Δφ, Δy ≈ 0 for 1 < p_T^ch < 2 GeV in 0–30 % Pb+Pb (2507.09307).

**Here the wake is the paired truth.** Every event's jet leg (MUSIC with the jet's deposition)
is hadronized against its own background leg (the same fireball without it), with correlated
sampling: ⟨bulk_jet⟩ − ⟨bulk_bg⟩ is the medium's response alone, with no mixed events. The jet
is the fragmentation of the surviving partons (`jet_frag`) without the Z's two leptons; wake + jet
is the full medium-modified Z-tagged yield.

**Inputs** (`zjet_wake.py` holds the helpers):

| | what | made by |
|---|---|---|
| `ZJET_HADRONS` | the Z+jet hadron files | `run_prod_jet.py --user-xml ../config/PbPb_0_30_ZJet.xml … --hadrons-only` (`utils/zjet/` launchers), or `--write-particlize` + `hadronize.py` |
| `ZJET_VACUUM` (optional) | the pp reference | `run_prod_vacuum.py` with the same XML and selection |
| `ZJET_PAIR` (optional) | one event's hydro evolution | `run_prod_jet.py … --write-particlize both` (not `--hadrons-only`) |
| `ZJET_MOVIE` (optional) | a rendered movie | `contribs/Visualization/wake_pyvista.py --file <ZJET_PAIR>` |

The notebook is stored without outputs. A small test set to run it on (96 Pb+Pb 0–30 % events,
6,000 vacuum events, one movie event) and the commands that make it are in `zjet_test/`; the
wake's significance needs the campaigns (`utils/zjet/README.md`).

In [ ]:
import os
import sys

import matplotlib.pyplot as plt
import numpy as np

try:
    import jetscape  # noqa: F401
except ImportError:              # PyJetscape not installed in this kernel: this checkout's
    sys.path.insert(0, os.path.abspath(os.path.join("..", "..", "python")))
import zjet_wake as zw
from jetscape.hadrons_h5 import HadronFileReader

HADRONS = os.environ.get("ZJET_HADRONS", "out_zjet")   # Z+jet hadron files (directory or glob)
VACUUM = os.environ.get("ZJET_VACUUM", "")             # run_prod_vacuum.py output (optional)
PAIR = os.environ.get("ZJET_PAIR", "")                 # a Z+jet pair file <stem>.h5 (optional)
PAIR_EVENT = int(os.environ.get("ZJET_PAIR_EVENT", "0"))
MOVIE = os.environ.get("ZJET_MOVIE", "")               # a wake_pyvista movie (optional)

r = HadronFileReader(HADRONS)
print(f"{r.n_events} events; correlated sampling: {r.correlated}; "
      f"pTHat windows: {None if r.pthat_bins is None else r.pthat_bins.tolist()}")

## 1. The event record

Every event carries its Z (`z_pt, z_y, z_phi, z_m` and the two leptons) and its N_coll. The
p̂T windows cut the hard scattering, not the Z: with initial-state radiation a Z of given p_T
comes from a spread of p̂T, which is why the production starts at p̂T = 20 GeV
(`utils/zjet/README.md`). Each event weighs its window's cross section over the window's events,
times its N_coll (Z production scales with it; the geometry is drawn minimum-bias-like over
0–30 %).

In [ ]:
t = zw.event_table(r)
has_z = np.isfinite(t["z_pt"])
print(f"{has_z.sum()} of {len(has_z)} events have a Z; N_coll {np.nanmin(t['n_coll']):.0f}"
      f" .. {np.nanmax(t['n_coll']):.0f}")

fig, ax = plt.subplots(1, 4, figsize=(18, 3.8))
bins = np.linspace(0, 250, 41)
for k in np.unique(t["pthat_bin"][has_z]):
    m = has_z & (t["pthat_bin"] == k)
    lo, hi = r.pthat_bins[k]
    ax[0].hist(t["z_pt"][m], bins, histtype="step", lw=1.6, label=f"p̂T {lo:g}–{hi:g}")
ax[0].set(xlabel="$p_T^Z$ [GeV]", ylabel="events", title="$p_T^Z$ per p̂T window")
ax[0].legend(fontsize=7)
w_sigma = zw.event_weights(t, has_z, ncoll=False)
ax[1].hist(t["z_pt"][has_z], bins, weights=w_sigma[has_z], histtype="stepfilled",
           color="0.6")
ax[1].set(yscale="log", xlabel="$p_T^Z$ [GeV]", ylabel="σ-weighted (a.u.)",
          title="cross-section weighted")
ax[2].hist(t["n_coll"][has_z], 20, color="0.6")
ax[2].set(xlabel="$N_{coll}$", ylabel="events", title="geometry (one value per background)")
ax[3].hist(t["z_m"][has_z], np.linspace(60, 120, 31), color="0.6")
ax[3].set(xlabel="$m_{\\ell\\ell}$ [GeV]", title="Z mass (e⁺e⁻ and μ⁺μ⁻)")
fig.tight_layout()

## 2. Selection and weights

The CMS 2507.09307 selection (`zw.CMS_2507`): 40 < p_T^Z < 350 GeV, |y_Z| < 2.4,
60 < m < 120 GeV, both leptons p_T > 20 GeV and |η| < 2.4; tracks |η| < 2.4. (CMS uses μμ only;
the flavour does not change the physics, so both channels are kept: `flavour=13` selects μμ.)
The weights vary by the window's cross section and by N_coll, so the effective number of
events is smaller than the count.

In [ ]:
SEL = zw.CMS_2507
sel = zw.select(t, **SEL)
w = zw.event_weights(t, sel)
EV = np.flatnonzero(sel)
print(f"selected {sel.sum()} events (effective {zw.effective_events(w, sel):.1f} after the "
      f"cross-section x N_coll weights); <p_T^Z> = {np.average(t['z_pt'][sel], weights=w[sel]):.1f} GeV")

## 3. The Z frame: (Δη, Δφ) maps

Per Z and per unit (Δη, Δφ), for the three track p_T classes of 2507.09307. Left: the wake
(paired ⟨bulk_jet⟩ − ⟨bulk_bg⟩, red = more hadrons than without the jet, blue = fewer);
middle: the jet's fragments; right: both. The Z points to (0, 0), the recoiling jet to Δφ = π
(dashed). One pass over the files fills all three classes.

In [ ]:
DETA = np.linspace(-4.0, 4.0, 21)                        # 0.4 in eta
DPHI = np.linspace(-np.pi / 2, 3 * np.pi / 2, 25)        # pi / 12
AREA = np.diff(DETA)[:, None] * np.diff(DPHI)[None, :]
CLASSES = list(zip(zw.PT_CLASSES[:-1], zw.PT_CLASSES[1:]))
Y = zw.z_frame_yields(r, EV, w, DETA, DPHI)
wake, wake_err = Y["wake"]
jet, jet_err = Y["jet"]
for c, (lo, hi) in enumerate(CLASSES):
    print(f"p_T^ch {lo:g}-{hi:g} GeV: wake {wake[c].sum():+.2f} +- "
          f"{np.sqrt((wake_err[c] ** 2).sum()):.2f}, jet {jet[c].sum():.2f} hadrons per Z")

In [ ]:
def zframe_axes(a, title):
    a.axhline(np.pi, color="k", ls="--", lw=0.8)
    a.plot([0], [0], marker="*", color="k", ms=10)
    a.text(0.15, 0.15, "Z", fontsize=10)
    a.text(0.15, np.pi + 0.15, "jet", fontsize=10)
    a.set(xlabel="Δη = η_h − y_Z", ylabel="Δφ = φ_h − φ_Z", title=title)

fig, ax = plt.subplots(len(CLASSES), 3, figsize=(16, 4.2 * len(CLASSES)), squeeze=False)
for c, (lo, hi) in enumerate(CLASSES):
    dw = wake[c] / AREA
    lim = np.nanpercentile(np.abs(dw), 98) or 1.0
    m = ax[c, 0].pcolormesh(DETA, DPHI, dw.T, cmap="RdBu_r", vmin=-lim, vmax=lim)
    fig.colorbar(m, ax=ax[c, 0], label="dN/dΔη dΔφ per Z")
    zframe_axes(ax[c, 0], f"wake, {lo:g} < p_T^ch < {hi:g} GeV")
    m = ax[c, 1].pcolormesh(DETA, DPHI, (jet[c] / AREA).T, cmap="viridis")
    fig.colorbar(m, ax=ax[c, 1], label="dN/dΔη dΔφ per Z")
    zframe_axes(ax[c, 1], f"jet fragments, {lo:g} < p_T^ch < {hi:g} GeV")
    tot = (wake[c] + jet[c]) / AREA
    lim = np.nanpercentile(np.abs(tot), 98) or 1.0
    m = ax[c, 2].pcolormesh(DETA, DPHI, tot.T, cmap="RdBu_r", vmin=-lim, vmax=lim)
    fig.colorbar(m, ax=ax[c, 2], label="dN/dΔη dΔφ per Z")
    zframe_axes(ax[c, 2], f"wake + jet, {lo:g} < p_T^ch < {hi:g} GeV")
fig.tight_layout()

## 4. Δφ distributions (as 2507.09307, Fig. 1)

Integrated over Δη (tracks |η| < 2.4), per Z. The wake's net hadron number need not vanish (the
deposited energy adds particles), so the dotted curve also shows it minus its Δφ average:
the shape CMS measures after subtracting ⟨N_ch⟩. A diffusion wake is a deficit at Δφ ≈ 0
relative to that average, the positive wake an excess at Δφ ≈ π.

In [ ]:
PHI_C = 0.5 * (DPHI[1:] + DPHI[:-1])
WPHI = np.diff(DPHI)
fig, ax = plt.subplots(1, len(CLASSES), figsize=(5.5 * len(CLASSES), 4), squeeze=False)
for c, (lo, hi) in enumerate(CLASSES):
    a = ax[0, c]
    hw, ew = zw.project(wake[c], wake_err[c], 0)
    hj, ej = zw.project(jet[c], jet_err[c], 0)
    a.errorbar(PHI_C, hw / WPHI, ew / WPHI, fmt="o", color="C3", ms=4, label="wake")
    a.errorbar(PHI_C, hj / WPHI, ej / WPHI, fmt="s", color="C0", ms=4, mfc="none",
               label="jet fragments")
    a.errorbar(PHI_C, (hw + hj) / WPHI, np.hypot(ew, ej) / WPHI, fmt="-", color="k",
               lw=1.2, label="wake + jet")
    a.plot(PHI_C, (hw - np.average(hw, weights=WPHI)) / WPHI, ":", color="C3",
           label="wake − ⟨wake⟩")
    a.axhline(0, color="0.5", lw=0.6)
    a.axvline(0, color="0.7", lw=0.6)
    a.axvline(np.pi, color="0.7", lw=0.6, ls="--")
    a.set(xlabel="Δφ = φ_h − φ_Z", ylabel="dN/dΔφ per Z",
          title=f"{lo:g} < p_T^ch < {hi:g} GeV")
    a.legend(fontsize=8)
fig.tight_layout()

## 5. Δη on the Z side (as 2507.09307, Fig. 2)

Only |Δφ| < π/2: the region behind the jet. The diffusion wake is a dip at Δη ≈ 0, strongest
for 1 < p_T^ch < 2 GeV; the medium's flow sets the width.

In [ ]:
ETA_C = 0.5 * (DETA[1:] + DETA[:-1])
WETA = np.diff(DETA)
zside = np.abs(PHI_C) < np.pi / 2
fig, ax = plt.subplots(1, len(CLASSES), figsize=(5.5 * len(CLASSES), 4), squeeze=False)
for c, (lo, hi) in enumerate(CLASSES):
    a = ax[0, c]
    hw, ew = zw.project(wake[c], wake_err[c], 1, zside)
    hj, ej = zw.project(jet[c], jet_err[c], 1, zside)
    a.errorbar(ETA_C, hw / WETA, ew / WETA, fmt="o", color="C3", ms=4, label="wake, |Δφ| < π/2")
    a.errorbar(ETA_C, hj / WETA, ej / WETA, fmt="s", color="C0", ms=4, mfc="none",
               label="jet fragments, |Δφ| < π/2")
    a.axhline(0, color="0.5", lw=0.6)
    a.set(xlabel="Δη = η_h − y_Z", ylabel="dN/dΔη per Z", title=f"{lo:g} < p_T^ch < {hi:g} GeV")
    a.legend(fontsize=8)
fig.tight_layout()

## 6. Pb+Pb − pp (as 2507.09307 Fig. 3, 2103.04377 Fig. 1)

The vacuum reference (`run_prod_vacuum.py`: the same PythiaGun, Matter in vacuum, Colorless,
no medium) gives the pp yield per Z with the same selection. Pb+Pb's wake + jet minus pp is the
experiments' observable; here both sides are truth, so the difference is the medium's
modification of the jet plus the wake, without any background subtraction.

The lower row shows the difference twice. **Absolute** (green): what the medium adds or removes
per Z; at 1–2 GeV the wake's extra soft hadrons lift it everywhere. **As CMS** (black): each
system minus its own Δφ average (what the mixed-event subtraction removes, over all Δη), ±Δφ
merged and halved, π/10 bins, which is 2507.09307's Fig. 3. The two differ by the absolute
difference's Δφ average and by the binning; `zjet_test/compare.py` compares the CMS form with
the HEPData tables.

In [ ]:
if VACUUM:
    rv = HadronFileReader(VACUUM)
    tv = zw.event_table(rv)
    selv = zw.select(tv, **SEL)
    wv = zw.event_weights(tv, selv)
    pp, pp_err = zw.vacuum_yields(rv, np.flatnonzero(selv), wv, DETA, DPHI)
    print(f"vacuum: {selv.sum()} selected events (effective {zw.effective_events(wv, selv):.0f})")
    # CMS's convention: pi/10 bins, all Delta eta (|eta_h|, |y_Z| < 2.4), +-Delta phi merged,
    # each system minus its Delta phi average
    DPHI_CMS = np.linspace(-np.pi / 2, 3 * np.pi / 2, 21)
    PHI_CMS = 0.5 * (DPHI_CMS[1:] + DPHI_CMS[:-1])
    yc = zw.z_frame_yields(r, EV, w, [-4.8, 4.8], DPHI_CMS)
    pb_cms = zw.cms_dphi((yc["wake"][0] + yc["jet"][0])[:, 0],
                         np.hypot(yc["wake"][1], yc["jet"][1])[:, 0], DPHI_CMS)
    vc = zw.vacuum_yields(rv, np.flatnonzero(selv), wv, [-4.8, 4.8], DPHI_CMS)
    pp_cms = zw.cms_dphi(vc[0][:, 0], vc[1][:, 0], DPHI_CMS)
    fig, ax = plt.subplots(2, len(CLASSES), figsize=(5.5 * len(CLASSES), 7), squeeze=False,
                           sharex=True)
    for c, (lo, hi) in enumerate(CLASSES):
        hw, ew = zw.project(wake[c], wake_err[c], 0)
        hj, ej = zw.project(jet[c], jet_err[c], 0)
        hp, ep = zw.project(pp[c], pp_err[c], 0)
        tot, etot = hw + hj, np.sqrt(ew ** 2 + ej ** 2)
        a = ax[0, c]
        a.errorbar(PHI_C, tot / WPHI, etot / WPHI, fmt="o", color="C3", ms=4,
                   label="Pb+Pb 0–30 % (wake + jet)")
        a.errorbar(PHI_C, hp / WPHI, ep / WPHI, fmt="s", color="k", ms=4, mfc="none",
                   label="pp (vacuum)")
        a.set(ylabel="dN/dΔφ per Z", title=f"{lo:g} < p_T^ch < {hi:g} GeV")
        a.legend(fontsize=8)
        b = ax[1, c]
        b.errorbar(PHI_C, (tot - hp) / WPHI, np.hypot(etot, ep) / WPHI, fmt="o", color="C2",
                   ms=4, label="absolute")
        b.errorbar(PHI_CMS, pb_cms[0][c] - pp_cms[0][c], np.hypot(pb_cms[1][c], pp_cms[1][c]),
                   fmt="d", color="k", ms=4, mfc="none",
                   label="as CMS: each − its Δφ average, ±Δφ merged, π/10")
        b.axhline(0, color="0.5", lw=0.6)
        b.set(xlabel="Δφ = φ_h − φ_Z", ylabel="Pb+Pb − pp, dN/dΔφ per Z")
        b.legend(fontsize=7)
    fig.tight_layout()
else:
    print("no ZJET_VACUUM: skipped")

## 7. The wake against p_T^Z

The Z side (|Δφ| < π/2, |Δη| < 1) and the jet side (|Δφ − π| < π/2) of the wake for
1 < p_T^ch < 2 GeV in bins of p_T^Z. A harder Z means a harder recoiling parton, more energy
lost and a stronger wake.

In [ ]:
ZPT = [(40, 60), (60, 100), (100, 350)]
inner = np.abs(ETA_C) < 1.0
jside = np.abs(PHI_C - np.pi) < np.pi / 2
rows = []
for lo, hi in ZPT:
    s = sel & (t["z_pt"] > lo) & (t["z_pt"] < hi)
    if s.sum() < 2:
        rows.append((lo, hi, s.sum(), np.nan, np.nan, np.nan, np.nan))
        continue
    ws = zw.event_weights(t, s)
    y = zw.z_frame_yields(r, np.flatnonzero(s), ws, DETA, DPHI, fragments=False)["wake"]
    h, e = y[0][0], y[1][0]                                  # 1 < p_T^ch < 2 GeV
    zs = h[np.ix_(inner, zside)].sum(), np.sqrt((e[np.ix_(inner, zside)] ** 2).sum())
    js = h[:, jside].sum(), np.sqrt((e[:, jside] ** 2).sum())
    rows.append((lo, hi, s.sum(), *zs, *js))
fig, ax = plt.subplots(figsize=(6, 4))
x = np.arange(len(rows))
ax.errorbar(x - 0.05, [r_[3] for r_ in rows], [r_[4] for r_ in rows], fmt="o", color="C0",
            label="Z side: |Δφ| < π/2, |Δη| < 1")
ax.errorbar(x + 0.05, [r_[5] for r_ in rows], [r_[6] for r_ in rows], fmt="s", color="C3",
            label="jet side: |Δφ − π| < π/2")
ax.axhline(0, color="0.5", lw=0.6)
ax.set_xticks(x, [f"{lo}–{hi}\n({n} ev.)" for lo, hi, n, *_ in rows])
ax.set(xlabel="$p_T^Z$ [GeV]", ylabel="wake hadrons per Z, 1 < p_T^ch < 2 GeV")
ax.legend(fontsize=8)
fig.tight_layout()

## 8. One event's hydro wake

With the hydro evolution stored (`--write-particlize both`, not `--hadrons-only`), the energy
density of the jet leg minus its background, in the transverse plane around the Z's rapidity
(|η_s − y_Z| < 0.5). The arrow from the hard vertex points along the Z, the dashed line along
the hardest parton (the recoiling jet). The deposition starts as the jet's sources; hydrodynamics
turns it into a front moving with the jet and a depleted region behind it, toward the Z.

In [ ]:
if PAIR:
    TAUS = [1.5, 3.0, 5.0, 8.0]
    x, y, frames, z, vtx, phi_lead = zw.transverse_wake(PAIR, PAIR_EVENT, TAUS)
    fig, ax = plt.subplots(1, len(frames), figsize=(4.6 * len(frames), 4.2))
    for a, (tau, de) in zip(np.atleast_1d(ax), frames.items()):
        lim = np.nanpercentile(np.abs(de), 99.5) or 1.0
        m = a.pcolormesh(x, y, de.T, cmap="RdBu_r", vmin=-lim, vmax=lim)
        a.set_aspect("equal")
        L = 6.0
        a.annotate("", xy=(vtx[0] + L * np.cos(z["z_phi"]), vtx[1] + L * np.sin(z["z_phi"])),
                   xytext=vtx, arrowprops=dict(arrowstyle="->", color="k", lw=1.8))
        a.text(vtx[0] + 1.1 * L * np.cos(z["z_phi"]), vtx[1] + 1.1 * L * np.sin(z["z_phi"]), "Z")
        if phi_lead is not None:
            a.plot([vtx[0], vtx[0] + L * np.cos(phi_lead)], [vtx[1], vtx[1] + L * np.sin(phi_lead)],
                   "k--", lw=1.2)
        a.set(xlim=(-12, 12), ylim=(-12, 12), xlabel="x [fm]", ylabel="y [fm]",
              title=f"τ = {tau:.1f} fm/c")
        fig.colorbar(m, ax=a, shrink=0.8, label="Δe [GeV/fm³]")
    fig.suptitle(f"event {PAIR_EVENT}: Z p_T = {z['z_pt']:.0f} GeV, y_Z = {z['z_y']:+.2f}")
    fig.tight_layout()
else:
    print("no ZJET_PAIR: skipped")

## 9. Movies

The jet-jet movie scripts in `contribs/Visualization/` work for Z+jet pair files. Their Z overlay
draws the Z's two leptons as straight tracks from the hard vertex (green) with an arrow along
the Z, and keeps them out of the partons and the jet hadrons (`--no-z` turns it off):

```bash
cd contribs/Visualization
# medium | medium + deposit | the wake, with the shower and the Z
python wake_pyvista.py --file <pair>.h5 --event 1 --outdir movies --movie zjet_wake.mp4
# the jet leg with its hadrons (needs hadronize.py on the particlize file)
python hydro_jet_particles_pyvista.py --file <pair>.h5 --event 1 --outdir movies
```

Below, one frame of such a movie (`ZJET_MOVIE`: the movie, or a PNG frame of it).

In [ ]:
img, what = None, ""
if MOVIE and os.path.exists(MOVIE):
    if MOVIE.lower().endswith((".png", ".jpg", ".jpeg")):       # a frame saved from a movie
        img, what = plt.imread(MOVIE), os.path.basename(MOVIE)
    else:
        try:
            import imageio.v3 as iio
            frames = iio.imread(MOVIE, index=None)
            k = int(0.55 * (len(frames) - 1))
            img, what = frames[k], f"{os.path.basename(MOVIE)}: frame {k} of {len(frames)}"
        except Exception as exc:          # imageio without its ffmpeg backend
            print(f"cannot read {MOVIE} here ({type(exc).__name__}; needs imageio[ffmpeg]): "
                  "pass a PNG frame as ZJET_MOVIE instead")
if img is not None:
    fig, ax = plt.subplots(figsize=(16, 7.2))
    ax.imshow(img)
    ax.axis("off")
    ax.set_title(what)
    fig.tight_layout()
elif not MOVIE:
    print("no ZJET_MOVIE: skipped")

## Caveats

* **Truth, not detector level.** No tracking efficiency, no lepton or jet reconstruction; the wake
  is the paired difference, not a mixed-event subtraction. The experiments' ⟨N_ch⟩ subtraction
  is only mimicked by the dotted curves of §4.
* **Centrality is geometric** (b < 8.57 fm for 0–30 %), not CMS's HF-energy percentiles, and the
  events are weighted by N_coll to give the Z-tagged centrality mix. With 200,000 minimum-bias
  3dMCGlauber events (`zjet_test/centrality_check.py`): the model's σ_inel is 7.89 b, so the cut
  is its 0–29.3 % (0–30 % exactly: 8.68 fm), and activity-based 0–30 % classes (string count,
  N_part) share 96–97 % of their events with it, with the same ⟨N_coll⟩.
* **MPI.** At 5.02 TeV Pythia hands over ~15 partons above 2 GeV per event besides the recoiling
  one; they shower and deposit too, adding a wake that is flat in Δφ (CMS discusses the same
  contribution in 2103.04377).
* **The Z's leptons** are in `jet_frag` (and pass `--charged`): every selection here removes them
  (`zw.not_z_lepton`); decay leptons of hadrons stay.
* **Statistics.** The test set of `zjet_test/` (96 events on 4 hydro backgrounds) is far below a
  campaign; read what it gives as the method, not the physics.